# 1. Setup & Import

In [1]:
import os, sys, json, torch, penman, numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from transformers import AutoConfig
from model_interface.modeling_bart import MBartForConditionalGeneration
from model_interface.tokenization_bart import AMRBartTokenizer
from common.postprocessing import ParsedStatus
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
print(f"Pytorch Version : {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

c:\D\ITB\riset_gnn\generate_amr\venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pytorch Version : 2.12.1+cu126
CUDA Available : True


# 2. Load Model & Tokenizer (AMR)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
print(f"Default Device : {device}")

Default Device : cuda


## Model AMR (Nafkhan)

In [3]:
AMR_MODEL_PATH = os.path.join(
    os.getcwd(),
    "..", "..", "..",
    "models",
    "mbart-en-id-smaller-concat-finetuned",
    "mbart-en-id-smaller-concat-finetuned"
)

amr_config = AutoConfig.from_pretrained(AMR_MODEL_PATH)
print(f"Model type : {amr_config.model_type}")
print(f"Model type : {amr_config.architectures}")
print(f"Model type : {amr_config.vocab_size}")

Model type : mbart
Model type : ['MBartForConditionalGeneration']
Model type : 38025


In [4]:
amr_tokenizer = AMRBartTokenizer.from_pretrained(AMR_MODEL_PATH, use_fast=False)
amr_model = MBartForConditionalGeneration.from_pretrained(AMR_MODEL_PATH, config=amr_config)
amr_model.resize_token_embeddings(len(amr_tokenizer))
amr_model = amr_model.to(device)

amr_model.eval()

print(f"Model Loaded on : {amr_model.device}")
print(f"Model Parameters : {sum([p.numel() for p in amr_model.parameters() if p.requires_grad is True])}")


[transformers] MBartForConditionalGeneration has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Added 0 AMR tokens


Loading weights: 100%|██████████| 519/519 [00:00<00:00, 41491.35it/s]
[transformers] MBartForConditionalGeneration has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Model Loaded on : cuda:0
Model Parameters : 432590848


# 3. Load Dataset (From CSV or jsonl)

In [5]:
def load_dataset(dataset : str, from_csv : bool):
    pass

# 4. Create Custom Dataset & Custom DataLoader
This is needed for batch processing also keep track ID of dataset

In [6]:
XLSUM_DATASET_PATH = os.path.join("..", "..", "..", "data", "xlsum")
# TODO do parser for liputan6 dataset
# LIPUTAN6_DATASET_PATH =

# Output directory for parsed AMR graphs
AMR_OUTPUT_DIR = os.path.join(XLSUM_DATASET_PATH, "amr_graphs")
os.makedirs(AMR_OUTPUT_DIR, exist_ok=True)

class XLSumDataset(Dataset):
    """Custom Dataset For XLSum Only
    
    Only includes rows where:
      1. A translation file exists in the 'translate' folder
      2. An AMR output file does NOT yet exist in the output folder
    
    This allows incremental processing: re-running the notebook will
    only parse IDs that have translations but haven't been parsed yet.
    """

    def __init__(self, path, output_dir=AMR_OUTPUT_DIR):
        df = pd.read_csv(os.path.join(XLSUM_DATASET_PATH, "analysis_data.csv"))

        translate_dir = os.path.join(XLSUM_DATASET_PATH, "translate")

        # Build a set of available translation IDs (filename without .txt)
        available_translations = set()
        if os.path.isdir(translate_dir):
            for fname in os.listdir(translate_dir):
                if fname.endswith(".txt"):
                    available_translations.add(fname[:-4])  # strip .txt

        # Build a set of already-parsed IDs
        already_parsed = set()
        if os.path.isdir(output_dir):
            for fname in os.listdir(output_dir):
                if fname.endswith(".txt"):
                    already_parsed.add(fname[:-4])  # strip .txt

        total_rows = len(df)

        # Filter: keep only rows with translation available AND not yet parsed
        mask_translated = df["id"].isin(available_translations)
        mask_not_parsed = ~df["id"].isin(already_parsed)
        self.df = df[mask_translated & mask_not_parsed].reset_index(drop=True)

        print(f"Total rows in CSV           : {total_rows}")
        print(f"Translations available      : {mask_translated.sum()}")
        print(f"Already parsed (skipped)    : {(mask_translated & ~mask_not_parsed).sum()}")
        print(f"Remaining to parse          : {len(self.df)}")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        filename = row["id"] + '.txt'

        with open(os.path.join(XLSUM_DATASET_PATH, "translate", filename), "r", encoding="utf-8") as f:
            translated = f.read()


        return {
            # here id is use so we don't lose track the data and can be used
            # as name of the file if we want to store all the amr graph on .txt
            "id" : row["id"],
            "text" : row["text"],
            "translated_text" : translated
        }

In [7]:
def add_amr_mask(tokenized_inputs, masks):
    """
    Appends <AMR> <mask> </AMR> special tokens to each tokenized input,
    and extends the attention masks accordingly.

    Args:
        tokenized_inputs: List[List[int]] - batch of input_ids
        masks:            List[List[int]] - batch of attention_masks (1s and 0s)

    Returns:
        (updated_inputs, updated_masks): both with AMR tokens appended before padding
    """
    amr_suffix = [
        amr_tokenizer.amr_bos_token_id,
        amr_tokenizer.mask_token_id,
        amr_tokenizer.amr_eos_token_id,
    ]
    amr_mask_suffix = [1, 1, 1]  # these tokens should always be attended to

    updated_inputs = []
    updated_masks = []

    for input_ids, mask in zip(tokenized_inputs, masks):
        # Find where real tokens end and padding begins
        # (padding tokens have mask == 0)
        pad_start = mask.index(0) if 0 in mask else len(mask)

        # Insert AMR tokens before the padding
        new_input_ids = input_ids[:pad_start] + amr_suffix + input_ids[pad_start:]
        new_mask     = mask[:pad_start]      + amr_mask_suffix + mask[pad_start:]

        updated_inputs.append(new_input_ids)
        updated_masks.append(new_mask)

    return updated_inputs, updated_masks


def wrapper_collate_fn(prefix_lang1, prefix_lang2):
    def xlsum_collate_fn(batch):
        all_ids, all_texts, all_translated_texts = [], [], []
        
        for i in batch:
            all_ids.append(i["id"])
            all_texts.append(i["text"])
            all_translated_texts.append(i["translated_text"])

        # concat with lang_prefix and <AMR> <mask> </AMR>

        all_inputs = []
        for text, translated_text in zip(all_texts, all_translated_texts):
            cur_result = f"{prefix_lang1} {text} {prefix_lang2} {translated_text}"
            all_inputs.append(cur_result)

        tokenized_inputs = amr_tokenizer(
            all_inputs, max_length = None, padding = True, truncation = True
        )

        updated_inputs, updated_masks = add_amr_mask(
            tokenized_inputs["input_ids"],
            tokenized_inputs["attention_mask"]
        )

        return all_ids, updated_inputs, updated_masks

    return xlsum_collate_fn

    
    

In [8]:
ds = XLSumDataset(XLSUM_DATASET_PATH)

loader = DataLoader(ds, 1, collate_fn=wrapper_collate_fn("id_ID", "en_XX"))

Total rows in CSV           : 47802
Translations available      : 47802
Already parsed (skipped)    : 2
Remaining to parse          : 47800


# 5. Parsing AMR and Store it On .txt Folder

In [9]:
def decode_amr_output(pred_token_ids, tokenizer):
    """
    Decode the raw model output token IDs into an AMR graph string.

    Args:
        pred_token_ids: List[int] - raw predicted token IDs from model.generate()
        tokenizer: AMRBartTokenizer instance

    Returns:
        amr_string: str - the penman-encoded AMR graph string
        status: ParsedStatus - parsing status (OK, FIXED, BACKOFF)
    """
    # Fix: ensure first token is bos
    pred_ids = list(pred_token_ids)
    pred_ids[0] = tokenizer.bos_token_id

    # Replace amr_eos with eos and remove padding
    pred_ids = [
        tokenizer.eos_token_id if tok == tokenizer.amr_eos_token_id else tok
        for tok in pred_ids
        if tok != tokenizer.pad_token_id
    ]

    # Decode into AMR graph
    graph, status, (nodes, backreferences) = tokenizer.decode_amr(
        pred_ids, restore_name_ops=False
    )

    # Encode the graph into a readable string
    amr_string = penman.encode(graph)

    return amr_string, status


def store_graph(ids, amr_strings, output_dir=AMR_OUTPUT_DIR):
    """
    Store AMR graph strings as individual .txt files.

    Args:
        ids: List[str] - list of data IDs (used as filenames)
        amr_strings: List[str] - list of AMR graph strings
        output_dir: str - directory to save the .txt files
    """
    os.makedirs(output_dir, exist_ok=True)

    for data_id, amr_str in zip(ids, amr_strings):
        filepath = os.path.join(output_dir, f"{data_id}.txt")
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(amr_str)

In [10]:
# Counters for parsing status summary
status_counts = {"OK": 0, "FIXED": 0, "BACKOFF": 0, "ERROR": 0}
total_parsed = 0

print(f"Starting AMR parsing for {len(ds)} samples...")
print(f"Output directory: {AMR_OUTPUT_DIR}")
print(f"{'='*60}\n")

for batch_ids, inputs, masks in tqdm(loader, desc="Parsing AMR"):
    try:
        with torch.no_grad():
            outputs = amr_model.generate(
                input_ids=torch.tensor(inputs, dtype=torch.long).to(device),
                attention_mask=torch.tensor(masks, dtype=torch.long).to(device),
                num_beams=5,
                max_length=1024,
                decoder_start_token_id=amr_tokenizer.amr_bos_token_id
            )

        # Decode each sample in the batch
        amr_strings = []
        for i in range(outputs.shape[0]):
            pred_ids = outputs[i].cpu().tolist()
            amr_string, status = decode_amr_output(pred_ids, amr_tokenizer)
            amr_strings.append(amr_string)

            # Track status
            status_name = status.name if hasattr(status, "name") else str(status)
            if status_name in status_counts:
                status_counts[status_name] += 1
            else:
                status_counts["ERROR"] += 1

        # Store the parsed AMR graphs
        store_graph(batch_ids, amr_strings)
        total_parsed += len(batch_ids)

    except Exception as e:
        # print(f"\n[ERROR] Failed to parse IDs {batch_ids}: {e}")
        status_counts["ERROR"] += len(batch_ids)
        continue

# Print summary
print(f"\n{'='*60}")
print(f"PARSING COMPLETE")
print(f"{'='*60}")
print(f"Total parsed : {total_parsed}")
for status_name, count in status_counts.items():
    if count > 0:
        pct = count / max(total_parsed, 1) * 100
        print(f"  {status_name:8s} : {count} ({pct:.1f}%)")
print(f"\nOutput saved to: {AMR_OUTPUT_DIR}")

Starting AMR parsing for 47800 samples...
Output directory: ..\data\xlsum\amr_graphs



Parsing AMR:   2%|▏         | 1148/47800 [00:44<29:54, 25.99it/s]


KeyboardInterrupt: 